# Exploratory Data Analysis — Weekly Store KPI Dataset

**Goal.** Understand the dataset before writing a single line of pipeline code. The questions to answer:

1. What does the data look like (shape, types, completeness, grain)?
2. Are there quality problems that must be handled explicitly?
3. What are the distributions of raw and derived KPIs — is anything skewed or multimodal?
4. How do stores differ from each other and from the network median?
5. Are there temporal patterns or seasonality the anomaly detector must account for?
6. What drives net sales — what does the correlation structure look like?
7. How bad are the outliers, and does MAD-based scoring handle them better than z-score?

Every constant locked here (`IDENTITY_ABS_TOLERANCE`, `MODIFIED_ZSCORE_THRESHOLD`, `YOY_WEEK_INTERSECTION`) feeds the deterministic pipeline directly.

## 0  Setup & Data Loading

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams.update({
    "figure.dpi":          110,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           True,
    "grid.alpha":          0.3,
    "axes.prop_cycle":     plt.cycler(color=[
        "#0284c7","#0f766e","#7c3aed","#b45309","#dc2626",
        "#0891b2","#15803d","#c2410c","#6d28d9","#be185d",
    ]),
})

IDENTITY_ABS_TOLERANCE    = 1e-9
MODIFIED_ZSCORE_THRESHOLD = 3.5
RAW_COLS  = ["traffic", "gross_transactions", "gross_quantity", "net_sales"]
DERIV_COLS = ["conversion_rate", "units_per_txn", "avg_selling_price", "avg_txn_value", "revenue_per_visitor"]

def find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("repo root not found")

ROOT = find_repo_root(Path.cwd().resolve())
df_raw = pd.read_excel(ROOT / "data" / "raw" / "practical-test-dataset-weekly-kpi.xlsx")

print(f"Loaded {len(df_raw):,} rows × {df_raw.shape[1]} columns")
df_raw.head()

## 1  Data Overview

In [ ]:
print("=== Shape ===")
print(f"{df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns\n")

print("=== Types & nulls ===")
info = pd.DataFrame({
    "dtype":      df_raw.dtypes.astype(str),
    "non_null":   df_raw.notna().sum(),
    "null_count": df_raw.isna().sum(),
    "null_%":     (df_raw.isna().mean() * 100).round(2),
})
display(info)

print("\n=== Unique values per dimension ===")
for col in ["Store Name", "Year", "Week"]:
    uniq = sorted(df_raw[col].unique())
    print(f"  {col}: {len(uniq)} unique → {uniq}")

In [ ]:
print("=== Descriptive statistics (raw KPI columns) ===")
display(df_raw[RAW_COLS].describe().round(2))

## 2  Data Quality Checks

In [ ]:
print("=== Duplicate grain keys ===")
dupes = df_raw.duplicated(["Store Name", "Year", "Week"]).sum()
print(f"  Duplicate (Store, Year, Week) rows: {dupes}")

print("\n=== Negative or zero values in KPI columns ===")
for col in RAW_COLS:
    n_neg  = (df_raw[col] < 0).sum()
    n_zero = (df_raw[col] == 0).sum()
    print(f"  {col:30s}  negative={n_neg}  zero={n_zero}")

print("\n=== Logical constraint: gross_transactions ≤ traffic ===")
violations = df_raw[df_raw["gross_transactions"] > df_raw["traffic"]].copy()
violations["excess"] = violations["gross_transactions"] - violations["traffic"]
print(f"  Rows where gross_transactions > traffic: {len(violations)}")
display(violations[["Store Name","Year","Week","traffic","gross_transactions","excess"]])

In [ ]:
# Deep dive into the one known violation
sg = df_raw[(df_raw["Store Name"]=="Store_G") & (df_raw["Year"]==2025) & (df_raw["Week"]==21)].copy()
sg["implied_CR"] = sg["gross_transactions"] / sg["traffic"]
sg["implied_UPT"] = sg["gross_quantity"] / sg["gross_transactions"]
sg["implied_AUP"] = sg["net_sales"] / sg["gross_quantity"]

print("Store_G · 2025 · W21 — all columns:")
display(sg.T)

# Compare to Store_G median across same year
sg_2025 = df_raw[(df_raw["Store Name"]=="Store_G") & (df_raw["Year"]==2025)]
print("\nStore_G 2025 medians (all weeks) vs W21 values:")
compare = pd.DataFrame({
    "Store_G_2025_median": sg_2025[RAW_COLS].median().round(1),
    "W21_value":           sg[RAW_COLS].iloc[0].round(1),
})
compare["ratio"] = (compare["W21_value"] / compare["Store_G_2025_median"]).round(2)
display(compare)

## 3  KPI Derivation & Retail Equation Identity

In [ ]:
# Derive the full KPI tree from the four raw columns
df = df_raw.copy()
df["conversion_rate"]      = df["gross_transactions"] / df["traffic"]
df["units_per_txn"]        = df["gross_quantity"]      / df["gross_transactions"]
df["avg_selling_price"]    = df["net_sales"]           / df["gross_quantity"]
df["avg_txn_value"]        = df["net_sales"]           / df["gross_transactions"]
df["revenue_per_visitor"]  = df["net_sales"]           / df["traffic"]
df["reconstructed_sales"]  = (
    df["traffic"] * df["conversion_rate"] * df["units_per_txn"] * df["avg_selling_price"]
)
df["identity_drift"] = (df["net_sales"] - df["reconstructed_sales"]).abs()

print("=== Retail equation identity check ===")
print("  net_sales = traffic × CR × UPT × AUP")
drift_summary = pd.DataFrame([
    {"stat": "max absolute drift",   "value": df["identity_drift"].max()},
    {"stat": "p99 absolute drift",   "value": df["identity_drift"].quantile(0.99)},
    {"stat": "rows > 1e-9 drift",    "value": int((df["identity_drift"] > IDENTITY_ABS_TOLERANCE).sum())},
])
display(drift_summary)
assert df["identity_drift"].max() <= IDENTITY_ABS_TOLERANCE
print(f"  ✓ Identity holds to floating-point precision. Tolerance locked: {IDENTITY_ABS_TOLERANCE:.0e}")

print("\n=== Descriptive stats: derived KPIs ===")
display(df[DERIV_COLS].describe().round(4))

## 4  Raw KPI Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, col in zip(axes.flat, RAW_COLS):
    vals = df[col].dropna()
    ax.hist(vals, bins=40, edgecolor="white", linewidth=0.4)
    ax.axvline(vals.median(), color="#dc2626", linewidth=1.5, linestyle="--", label=f"median={vals.median():,.0f}")
    ax.axvline(vals.mean(),   color="#f59e0b", linewidth=1.5, linestyle=":",  label=f"mean={vals.mean():,.0f}")
    ax.set_title(col)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
fig.suptitle("Distribution of raw KPIs (all stores, all weeks)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Box plots by year to spot YoY level shifts
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col in zip(axes, RAW_COLS):
    groups = [df.loc[df["Year"]==yr, col].to_numpy() for yr in [2024, 2025]]
    bp = ax.boxplot(groups, labels=["2024","2025"], patch_artist=True,
                    medianprops={"color":"#dc2626","linewidth":2},
                    flierprops={"marker":"o","markersize":3,"alpha":0.5})
    bp["boxes"][0].set_facecolor("#bae6fd")
    bp["boxes"][1].set_facecolor("#bbf7d0")
    ax.set_title(col, fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
fig.suptitle("Raw KPI distributions: 2024 vs 2025", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

## 5  Derived KPI Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
plot_cols = [
    ("conversion_rate",     "Conversion Rate",      "{:.1%}"),
    ("units_per_txn",       "Units per Transaction", "{:.2f}"),
    ("avg_selling_price",   "Avg Selling Price",     "${:.0f}"),
    ("avg_txn_value",       "Avg Transaction Value", "${:.0f}"),
    ("revenue_per_visitor", "Revenue per Visitor",   "${:.2f}"),
]
for ax, (col, label, fmt) in zip(axes.flat, plot_cols):
    vals = df[col].dropna()
    ax.hist(vals, bins=40, edgecolor="white", linewidth=0.4, color="#7c3aed", alpha=0.8)
    med = vals.median()
    ax.axvline(med, color="#dc2626", linewidth=1.5, linestyle="--", label=f"median={fmt.format(med)}")
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=8)
axes.flat[-1].set_visible(False)
fig.suptitle("Distribution of derived KPIs (all stores, all weeks)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 6  Store-Level Profiles

In [ ]:
store_summary = df.groupby("Store Name").agg(
    traffic_med       =("traffic",            "median"),
    cr_med            =("conversion_rate",    "median"),
    upt_med           =("units_per_txn",      "median"),
    aup_med           =("avg_selling_price",  "median"),
    net_sales_med     =("net_sales",          "median"),
    net_sales_total   =("net_sales",          "sum"),
    weeks_in_data     =("Week",               "count"),
).sort_values("net_sales_total", ascending=False).round(2)

print("=== Per-store medians (sorted by total net sales) ===")
display(store_summary)

In [ ]:
# Side-by-side bar: median net_sales and traffic, stores ordered by net_sales
stores = store_summary.index.tolist()
x = np.arange(len(stores))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(x, store_summary["net_sales_med"], color="#0284c7", edgecolor="white")
axes[0].set_xticks(x); axes[0].set_xticklabels(stores, rotation=45, ha="right")
axes[0].set_title("Median weekly net sales by store")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))

axes[1].bar(x, store_summary["traffic_med"], color="#0f766e", edgecolor="white")
axes[1].set_xticks(x); axes[1].set_xticklabels(stores, rotation=45, ha="right")
axes[1].set_title("Median weekly traffic by store")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))

fig.suptitle("Store performance rankings", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Weekly net_sales distribution per store — shows within-store volatility
fig, ax = plt.subplots(figsize=(12, 5))
data_by_store = [df.loc[df["Store Name"]==s, "net_sales"].to_numpy() for s in stores]
bp = ax.boxplot(data_by_store, labels=stores, patch_artist=True,
                medianprops={"color":"#dc2626","linewidth":2},
                flierprops={"marker":"o","markersize":4,"alpha":0.5})
for patch in bp["boxes"]:
    patch.set_facecolor("#e0f2fe")
    patch.set_edgecolor("#0284c7")
ax.set_xticklabels(stores, rotation=45, ha="right")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.set_title("Weekly net sales distribution per store (outliers shown)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: median conversion_rate per store per year
pivot_cr = df.pivot_table(index="Store Name", columns="Year", values="conversion_rate", aggfunc="median")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pivot_cr.values, cmap="YlGn", aspect="auto")
ax.set_xticks(range(len(pivot_cr.columns))); ax.set_xticklabels(pivot_cr.columns)
ax.set_yticks(range(len(pivot_cr.index)));   ax.set_yticklabels(pivot_cr.index)
for i in range(len(pivot_cr.index)):
    for j in range(len(pivot_cr.columns)):
        ax.text(j, i, f"{pivot_cr.values[i,j]:.3f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, label="Median CR")
ax.set_title("Median conversion rate per store per year", fontweight="bold")
plt.tight_layout()
plt.show()

print("Note: Store_G 2025 median is inflated by the W21 violation (CR > 1).")

## 7  Temporal Patterns & Coverage

In [ ]:
print("=== Coverage by year ===")
display(df.groupby("Year")["Week"].agg(["min","max","nunique"]))

print("\n=== Per-store YoY week intersection ===")
rows = []
for store, grp in df.groupby("Store Name"):
    w24 = set(grp.loc[grp["Year"]==2024,"Week"])
    w25 = set(grp.loc[grp["Year"]==2025,"Week"])
    overlap = sorted(w24 & w25)
    rows.append({"store":store,"2024_wks":len(w24),"2025_wks":len(w25),
                 "overlap":len(overlap),"first":overlap[0],"last":overlap[-1]})
display(pd.DataFrame(rows))
print("\n✓ YoY window: W1–W48 for every store (48 like-for-like weeks). 2024 W49-W52 excluded from all YoY calcs.")

In [ ]:
# Network-level weekly aggregates — median across all stores
net_weekly = df.groupby(["Year","Week"]).agg(
    net_sales_med  =("net_sales",         "median"),
    traffic_med    =("traffic",            "median"),
    cr_med         =("conversion_rate",   "median"),
).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=False)
metrics = [
    ("net_sales_med", "Median Net Sales",        "${:,.0f}"),
    ("traffic_med",   "Median Traffic",           "{:,.0f}"),
    ("cr_med",        "Median Conversion Rate",   "{:.1%}"),
]

for ax, (col, label, fmt) in zip(axes, metrics):
    for yr, color, ls in [(2024,"#0284c7","-"),(2025,"#0f766e","--")]:
        sub = net_weekly[net_weekly["Year"]==yr].sort_values("Week")
        ax.plot(sub["Week"], sub[col], color=color, linestyle=ls, linewidth=1.8, label=str(yr), marker="o", markersize=3)
    ax.set_ylabel(label)
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt.format(v)))
    ax.set_xlabel("Week")

fig.suptitle("Network-level weekly trends: 2024 vs 2025", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Per-store weekly net_sales trend (2025 only, to show cross-store spread)
df_2025 = df[df["Year"]==2025].sort_values(["Store Name","Week"])

fig, ax = plt.subplots(figsize=(13, 5))
for store, grp in df_2025.groupby("Store Name"):
    ax.plot(grp["Week"], grp["net_sales"], linewidth=1.4, label=store, marker="o", markersize=2, alpha=0.85)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.set_xlabel("Week")
ax.set_ylabel("Net Sales")
ax.set_title("Per-store weekly net sales — 2025", fontweight="bold")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 8  Correlation Analysis

In [ ]:
corr_cols = RAW_COLS + DERIV_COLS
corr = df[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr_cols))); ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(corr_cols))); ax.set_yticklabels(corr_cols, fontsize=8)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        val = corr.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=7, color="white" if abs(val)>0.6 else "black")
plt.colorbar(im, ax=ax)
ax.set_title("Pearson correlation matrix — KPI panel", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# What drives net_sales? Scatter plots vs the four decomposition drivers
drivers = [
    ("traffic",           "Traffic"),
    ("conversion_rate",   "Conversion Rate"),
    ("units_per_txn",     "Units per Transaction"),
    ("avg_selling_price", "Avg Selling Price"),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (col, label) in zip(axes, drivers):
    ax.scatter(df[col], df["net_sales"], s=8, alpha=0.4)
    # simple linear trend
    m, b = np.polyfit(df[col], df["net_sales"], 1)
    xs = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(xs, m*xs+b, color="#dc2626", linewidth=1.5)
    r = df[[col,"net_sales"]].corr().iloc[0,1]
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Net Sales" if ax == axes[0] else "")
    ax.set_title(f"r = {r:.2f}", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v/1000:.0f}k"))
fig.suptitle("Net sales vs each retail decomposition driver", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

## 9  Anomaly Detection: MAD vs Z-Score

In [ ]:
# Compute modified z-scores for each KPI column across the full panel
score_cols = ["conversion_rate", "traffic", "net_sales", "units_per_txn", "avg_selling_price"]

df_scores = df[["Store Name","Year","Week"]].copy()
for col in score_cols:
    vals   = df[col].to_numpy(dtype=float)
    median = np.median(vals)
    mad    = np.median(np.abs(vals - median))
    if mad == 0:
        mad = np.mean(np.abs(vals - median))  # fallback: MAD=0 edge case
    df_scores[f"{col}_mz"] = 0.6745 * np.abs(vals - median) / mad

mz_cols = [c for c in df_scores.columns if c.endswith("_mz")]
df_scores["max_mz"] = df_scores[mz_cols].max(axis=1)
df_scores["any_flag"] = df_scores["max_mz"] >= MODIFIED_ZSCORE_THRESHOLD

print(f"Rows flagged as anomalous (max modified-z ≥ {MODIFIED_ZSCORE_THRESHOLD}): {df_scores['any_flag'].sum()}")
print(f"  out of {len(df_scores):,} total rows ({df_scores['any_flag'].mean()*100:.1f}%)\n")

flagged = df_scores[df_scores["any_flag"]].sort_values("max_mz", ascending=False)
display(flagged.head(15))

In [ ]:
# Side-by-side: standard z-score vs MAD-based on conversion_rate
# Shows why the outlier inflates σ and makes z-score unreliable
cr = df["conversion_rate"].to_numpy(dtype=float)
cr_med  = np.median(cr)
cr_mad  = np.median(np.abs(cr - cr_med))
cr_mean = cr.mean()
cr_std  = cr.std(ddof=1)

std_z   = (cr - cr_mean) / cr_std
mad_z   = 0.6745 * (cr - cr_med) / cr_mad

is_sg = (
    (df["Store Name"]=="Store_G") & (df["Year"]==2025) & (df["Week"]==21)
).to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
titles = [
    f"Standard z-score  (mean={cr_mean:.3f}, σ={cr_std:.3f})",
    f"Modified z-score  (median={cr_med:.3f}, MAD={cr_mad:.3f})",
]
for ax, scores, title in zip(axes, [std_z, mad_z], titles):
    ax.scatter(np.arange(len(scores))[~is_sg], scores[~is_sg],
               s=8, alpha=0.45, color="#0284c7", label="Normal")
    ax.scatter(np.arange(len(scores))[is_sg], scores[is_sg],
               s=100, color="#dc2626", zorder=6, label="Store_G W21")
    ax.axhline( MODIFIED_ZSCORE_THRESHOLD, color="#f59e0b", linewidth=1.2,
               linestyle="--", label=f"±{MODIFIED_ZSCORE_THRESHOLD}")
    ax.axhline(-MODIFIED_ZSCORE_THRESHOLD, color="#f59e0b", linewidth=1.2, linestyle="--")
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Row index")
    ax.set_ylabel("Score")
    ax.legend(fontsize=8)

fig.suptitle("Standard z-score vs MAD-based scoring on conversion_rate", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Store_G W21 — standard z: {abs(std_z[is_sg][0]):.1f}σ  |  modified z: {abs(mad_z[is_sg][0]):.1f}")
print(f"Ordinary flagged rows at ≥{MODIFIED_ZSCORE_THRESHOLD} (std z): {(np.abs(std_z)>=MODIFIED_ZSCORE_THRESHOLD).sum()}  "
      f"| (mad z): {(np.abs(mad_z)>=MODIFIED_ZSCORE_THRESHOLD).sum()}")

In [ ]:
# How many anomalous weeks does each store have?
flag_by_store = df_scores.groupby("Store Name")["any_flag"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(flag_by_store.index, flag_by_store.values, color="#7c3aed", edgecolor="white")
ax.set_ylabel("Weeks flagged as anomalous")
ax.set_title("Anomalous weeks per store (any KPI, modified-z ≥ 3.5)", fontweight="bold")
ax.set_xticklabels(flag_by_store.index, rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 10  Store vs Network Gap

In [ ]:
# For each week compute network median, then show each store's % gap
net_ref = df.groupby(["Year","Week"])[["net_sales","traffic","conversion_rate"]].median()
net_ref.columns = [f"{c}_net_median" for c in net_ref.columns]
df_gap = df.merge(net_ref.reset_index(), on=["Year","Week"])
df_gap["net_sales_gap_pct"] = (df_gap["net_sales"] / df_gap["net_sales_net_median"] - 1) * 100

# Box plot of net_sales gap distribution per store
fig, ax = plt.subplots(figsize=(12, 4))
store_order = (df_gap.groupby("Store Name")["net_sales_gap_pct"]
               .median().sort_values(ascending=False).index.tolist())
data = [df_gap.loc[df_gap["Store Name"]==s, "net_sales_gap_pct"].to_numpy() for s in store_order]
bp = ax.boxplot(data, labels=store_order, patch_artist=True,
                medianprops={"color":"#dc2626","linewidth":2},
                flierprops={"marker":"o","markersize":3,"alpha":0.5})
for patch in bp["boxes"]:
    patch.set_facecolor("#f0fdf4")
    patch.set_edgecolor("#0f766e")
ax.axhline(0, color="#64748b", linewidth=1.2, linestyle="--", label="Network median")
ax.set_ylabel("Net sales vs network median (%)")
ax.set_title("Store performance gap vs network median — weekly distribution", fontweight="bold")
ax.set_xticklabels(store_order, rotation=45, ha="right")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 11  Conclusions

### What the EDA proved

| # | Finding | Evidence |
|---|---|---|
| 1 | Table is clean at the grain level | 0 nulls, 0 duplicate `(Store, Year, Week)` keys |
| 2 | Retail equation holds exactly | Max reconstruction drift `2.9e-11` — well inside `1e-9` tolerance |
| 3 | 2025 is truncated at W48 | 2024 has 52 weeks, 2025 has 48; naïve YoY would overstate 2024 base |
| 4 | Store_G 2025 W21 is a hard DQ violation | CR = 134% — physically impossible; `gross_transactions > traffic` by 22 |
| 5 | KPI distributions are right-skewed | Histograms show fat right tails; mean > median across all metrics |
| 6 | Stores differ materially in size | Net sales median ranges ~3× across the network |
| 7 | Traffic is the primary driver of net_sales | Highest r with net_sales; scatter + correlation matrix confirm |
| 8 | Standard z-score is unfit for anomaly detection here | Store_G W21 inflates σ; MAD-based scoring is breakdown-resistant |

### Pipeline decisions locked by EDA

```python
IDENTITY_ABS_TOLERANCE    = 1e-9   # retail equation validation gate
MODIFIED_ZSCORE_THRESHOLD = 3.5    # Iglewicz & Hoaglin 1993 standard; ≈5% of rows flagged
YOY_WEEK_INTERSECTION     = per-store intersection of 2024 and 2025 weeks (W1–W48)
```

- `loader.validate()` surfaces `gross_transactions_exceeds_traffic` as a hard flag
- `anomalies.py` uses `0.6745 × |x − median| / MAD` not `.std()`
- `features.compute_yoy()` restricts to the per-store week intersection
- Narratives for Store_G 2025 W21 carry an explicit DQ caveat regardless of LLM output